In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os

# ---------------------------------------------------------
# 1. Configuration & Data Setup
# ---------------------------------------------------------
folder_path = "01-29_Colors Baseline"

# Concentration Mapping (Row A -> H : 12.5% -> 100%)
concentration_map = {
    'A': 12.5, 'B': 25.0, 'C': 37.5, 'D': 50.0,
    'E': 62.5, 'F': 75.0, 'G': 87.5, 'H': 100.0
}

# Color Mapping for 3 colors
dye_colors = {
    "Sky Blue":      "deepskyblue",
    "Grass Green":   "forestgreen",
    "Sunset Yellow": "gold"
}

# Column assignment (4 repeats per color across 12 columns)
# Sky Blue:      1, 4, 7, 10
# Grass Green:   2, 5, 8, 11
# Sunset Yellow: 3, 6, 9, 12
color_columns = {
    "Sky Blue":      ["1", "4", "7", "10"],
    "Grass Green":   ["2", "5", "8", "11"],
    "Sunset Yellow": ["3", "6", "9", "12"]
}

analysis_config = {
    "450nm": {
        "filename": "Colors_Baseline450nm.csv"
    },
    "562nm": {
        "filename": "Colors_Baseline562nm.csv"
    },
    "600nm": {
        "filename": "Colors_Baseline600nm.csv"
    },
    "650nm": {
        "filename": "Colors_Baseline650nm.csv"
    }
}


# ---------------------------------------------------------
# 2. Main Processing Loop
# ---------------------------------------------------------
for wave_key, config in analysis_config.items():
    file_name = config["filename"]
    full_path = os.path.join(folder_path, file_name)
    
    print(f"\n{'='*60}")
    print(f"Processing: {wave_key} ({file_name})")
    print(f"{'='*60}")

    # --- Load Data ---
    try:
        df = pd.read_csv(full_path, index_col=0)
    except FileNotFoundError:
        print(f"Skipping {file_name}: File not found.")
        continue

    # --- Preprocessing ---
    df = df.loc[df.index.isin(concentration_map.keys())]
    df = df.apply(pd.to_numeric, errors='coerce')
    df = df[~df.index.duplicated(keep='first')]

    # =========================================================
    # [Part A] Heatmap & Statistics
    # =========================================================
    print(f"--- Generating Heatmap ---")
    
    df_heatmap = df.copy()

    mean_val = np.nanmean(df_heatmap.values)
    std_val = np.nanstd(df_heatmap.values)
    cv_val = (std_val / mean_val) * 100 if mean_val != 0 else 0

    print(f"[Stats] Mean: {mean_val:.4f} | Std: {std_val:.4f} | CV: {cv_val:.2f}%")

    # Plot Heatmap
    plt.figure(figsize=(14, 6))
    sns.heatmap(df_heatmap, annot=True, fmt=".3f", cmap="YlGnBu")
    plt.title(f"Heatmap - {wave_key}", fontsize=14, fontweight='bold')
    
    # Save & Show Heatmap
    save_name_hm = f"Heatmap_{file_name.replace('.csv', '.png')}"
    plt.savefig(os.path.join(folder_path, save_name_hm), dpi=300, bbox_inches='tight')
    plt.show()  
    print(f"Saved: {save_name_hm}")

    # Store data for color-based plots (collect across all wavelengths)
    if 'all_wave_data' not in dir():
        all_wave_data = {}
    all_wave_data[wave_key] = df

# =========================================================
# [Part B] Linearity Plots — Per Color (all wavelengths)
# =========================================================
wavelength_colors = {
    "450nm": "goldenrod",
    "562nm": "tab:orange",
    "600nm": "steelblue",
    "650nm": "navy"
}

x_values = np.array([concentration_map[idx] for idx in ['A','B','C','D','E','F','G','H']])

for color_name, columns in color_columns.items():
    print(f"\n{'='*60}")
    print(f"Linearity Plot for {color_name}")
    print(f"{'='*60}")
    
    plt.figure(figsize=(10, 8))
    
    for wave_key, df in all_wave_data.items():
        # Get data from all 4 replicate columns for this color
        replicate_data = []
        for col in columns:
            if col in df.columns:
                replicate_data.append(df[col].values)
        
        if len(replicate_data) == 0:
            continue
        
        # Stack replicate data and calculate mean/std
        replicate_array = np.array(replicate_data)
        y_mean = np.nanmean(replicate_array, axis=0)
        y_std = np.nanstd(replicate_array, axis=0, ddof=1)
        
        # Clean data (remove NaN)
        mask = ~np.isnan(y_mean)
        x_clean = x_values[mask]
        y_mean_clean = y_mean[mask]
        y_std_clean = y_std[mask]

        if len(x_clean) < 2:
            continue

        # Linear Regression on mean values
        slope, intercept, r_value, p_value, std_err = stats.linregress(x_clean, y_mean_clean)
        r_squared = r_value ** 2
        fit_line = slope * x_clean + intercept
        
        wl_color = wavelength_colors.get(wave_key, 'gray')

        # Plot Scatter with Error Bars
        plt.errorbar(x_clean, y_mean_clean, yerr=y_std_clean,
                     fmt='o', label=f"{wave_key} (n=4)",
                     color=wl_color, markersize=10,
                     capsize=5, capthick=1.5, elinewidth=1.5,
                     ecolor=wl_color, markeredgecolor='black', markeredgewidth=0.8,
                     alpha=0.9, zorder=3)
        
        # Plot Regression Line
        plt.plot(x_clean, fit_line, linestyle='--', linewidth=2, 
                 color=wl_color, label=f"{wave_key} Fit ($R^2$={r_squared:.4f})", zorder=2)
        
        print(f"  {wave_key}: R² = {r_squared:.4f}, Slope = {slope:.6f}")

    plt.title(f"Linearity Check — {color_name} Dye", fontsize=18, fontweight='bold', pad=20)
    plt.xlabel("Concentration (%)", fontsize=14, labelpad=15)
    plt.ylabel("Optical Density (OD)", fontsize=14, labelpad=15)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.6, zorder=0)
    plt.axhline(y=3.0, color='black', linestyle='-', alpha=0.5, linewidth=2, zorder=1, label="Saturation Limit (OD 3.0)")
    plt.legend(fontsize=11, loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    
    save_name = f"Linearity_{color_name}.png"
    plt.savefig(os.path.join(folder_path, save_name), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_name}")